# 03 · Ejercicio 2: limpieza e integración con el tipo de cambio
Entrada: tablas `*_raw` de bronze y `silver.tipo_cambio`.
Salida: `silver.clientes`, `silver.cuentas`, `silver.transacciones_gtq` y `silver.dq_reporte`.

In [0]:
%run ./_config

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

DEPARTAMENTOS_GT = [
    "Alta Verapaz", "Baja Verapaz", "Chimaltenango", "Chiquimula", "El Progreso", "Escuintla",
    "Guatemala", "Huehuetenango", "Izabal", "Jalapa", "Jutiapa", "Petén", "Quetzaltenango", "Quiché",
    "Retalhuleu", "Sacatepéquez", "San Marcos", "Santa Rosa", "Sololá", "Suchitepéquez", "Totonicapán", "Zacapa",
]

hallazgos = []


def registrar(tabla, regla, df_afectados, llave, accion):
    """Agrega una regla de calidad al reporte con su conteo e identificadores."""
    ids = [str(r[0]) for r in df_afectados.select(llave).collect()]
    hallazgos.append((tabla, regla, len(ids), ", ".join(sorted(set(ids))), accion))


def deduplicar(df, tabla, llave):
    """Elimina filas idénticas en las columnas de negocio y reporta llaves repetidas con datos distintos."""
    negocio = [c for c in df.columns if not c.startswith("_")]
    orden = [F.col(c).asc_nulls_last() for c in df.columns]

    df = df.withColumn("_copia", F.row_number().over(Window.partitionBy(*negocio).orderBy(*orden)))
    registrar(tabla, "duplicado_exacto", df.filter("_copia > 1"), llave, "eliminado")
    df = df.filter("_copia = 1").drop("_copia")

    df = df.withColumn("_version", F.row_number().over(Window.partitionBy(llave).orderBy(*orden)))
    registrar(tabla, "llave_repetida_con_datos_distintos", df.filter("_version > 1"), llave,
              "se conserva una versión y se reporta")
    return df.filter("_version = 1").drop("_version")


def a_fecha(columna, formato):
    return F.expr(f"to_date(try_to_timestamp(trim({columna}), '{formato}'))")


def sin_auxiliares(df):
    return df.drop(*[c for c in df.columns if c.startswith("_")])

## Clientes

In [0]:
clientes = deduplicar(
    spark.table(f"{BRONZE}.clientes_raw").select(
        F.expr("try_cast(trim(cliente_id) AS INT)").alias("cliente_id"),
        F.trim("nombre").alias("nombre"),
        F.coalesce(a_fecha("fecha_alta", "yyyy-MM-dd"), a_fecha("fecha_alta", "dd/MM/yyyy")).alias("fecha_alta"),
        F.initcap(F.trim("departamento")).alias("departamento"),
        F.expr("coalesce(nullif(trim(segmento), ''), 'No definido')").alias("segmento"),
        F.col("fecha_alta").alias("_fecha_alta_orig"),
        F.col("departamento").alias("_departamento_orig"),
        F.col("segmento").alias("_segmento_orig"),
    ),
    "clientes", "cliente_id",
)

registrar("clientes", "fecha_alta_en_formato_dd/MM/yyyy",
          clientes.filter(a_fecha("_fecha_alta_orig", "yyyy-MM-dd").isNull() & F.col("fecha_alta").isNotNull()),
          "cliente_id", "convertida a DATE")
registrar("clientes", "fecha_alta_no_interpretable", clientes.filter(F.col("fecha_alta").isNull()),
          "cliente_id", "queda NULL")
registrar("clientes", "departamento_normalizado", clientes.filter(F.col("_departamento_orig") != F.col("departamento")),
          "cliente_id", "initcap(trim(departamento))")
registrar("clientes", "departamento_no_reconocido",
          clientes.filter(~F.coalesce(F.col("departamento").isin(DEPARTAMENTOS_GT), F.lit(False))),
          "cliente_id", "se conserva")
registrar("clientes", "segmento_faltante", clientes.filter(F.expr("nullif(trim(_segmento_orig), '') IS NULL")),
          "cliente_id", "se asigna 'No definido'")

clientes = sin_auxiliares(clientes)
display(clientes.orderBy("cliente_id"))

cliente_id,nombre,fecha_alta,departamento,segmento
1,Ana López,2025-01-15,Guatemala,Premium
2,Carlos Pérez,2025-02-15,Guatemala,Estándar
3,María Gómez,2025-03-01,Quetzaltenango,Estándar
4,Luis Ramírez,2025-03-20,Escuintla,No definido
5,Sofía Castillo,2025-04-02,Sacatepéquez,Premium


## Cuentas

In [0]:
cuentas = deduplicar(
    spark.table(f"{BRONZE}.cuentas_raw").select(
        F.expr("try_cast(trim(cuenta_id) AS INT)").alias("cuenta_id"),
        F.expr("try_cast(trim(cliente_id) AS INT)").alias("cliente_id"),
        F.upper(F.trim("tipo_cuenta")).alias("tipo_cuenta"),
        F.coalesce(a_fecha("fecha_apertura", "yyyy-MM-dd"), a_fecha("fecha_apertura", "dd/MM/yyyy")).alias("fecha_apertura"),
        F.upper(F.trim("estado")).alias("estado"),
    ),
    "cuentas", "cuenta_id",
)

cuentas = (
    cuentas.join(clientes.select("cliente_id", F.lit(True).alias("_cliente_existe")), "cliente_id", "left")
    .withColumn("flag_cliente_inexistente", F.col("_cliente_existe").isNull())
    .select("cuenta_id", "cliente_id", "tipo_cuenta", "fecha_apertura", "estado", "flag_cliente_inexistente")
)

registrar("cuentas", "cliente_id_inexistente", cuentas.filter("flag_cliente_inexistente"), "cuenta_id",
          "se conserva con flag_cliente_inexistente")
display(cuentas.orderBy("cuenta_id"))

cuenta_id,cliente_id,tipo_cuenta,fecha_apertura,estado,flag_cliente_inexistente
101,1,AHORRO,2025-01-15,ACTIVA,false
102,1,MONETARIA,2025-02-10,ACTIVA,false
103,2,AHORRO,2025-02-15,ACTIVA,false
104,3,AHORRO,2025-03-01,CERRADA,false
105,4,AHORRO,2025-03-20,ACTIVA,false
106,9,AHORRO,2025-05-01,ACTIVA,true


## Transacciones en quetzales
`monto_gtq = abs(monto) × tasa_aplicada`: tasa 1 para GTQ y la del Banguat del día para USD, redondeado a centavos.

In [0]:
transacciones = deduplicar(
    spark.table(f"{BRONZE}.transacciones_raw").select(
        F.trim("transaccion_id").alias("transaccion_id"),
        F.expr("try_cast(trim(cuenta_id) AS INT)").alias("cuenta_id"),
        F.expr("try_to_timestamp(trim(fecha_hora), 'yyyy-MM-dd HH:mm:ss')").alias("fecha_hora"),
        F.expr("try_cast(trim(monto) AS DECIMAL(18,2))").alias("monto"),
        F.upper(F.trim("moneda")).alias("moneda"),
        F.upper(F.trim("tipo")).alias("tipo"),
        F.upper(F.trim("canal")).alias("canal"),
        F.upper(F.trim("estado")).alias("estado"),
        F.col("monto").alias("_monto_orig"),
        F.col("moneda").alias("_moneda_orig"),
    ),
    "transacciones", "transaccion_id",
)

tasas_usd = spark.table(f"{SILVER}.tipo_cambio").select("fecha", F.col("tasa").alias("_tasa_usd"))
cuentas_existentes = cuentas.select("cuenta_id", F.lit(True).alias("_cuenta_existe"))

transacciones_gtq = (
    transacciones.withColumn("fecha", F.to_date("fecha_hora"))
    .join(tasas_usd, "fecha", "left")
    .join(cuentas_existentes, "cuenta_id", "left")
    .withColumn("tasa_aplicada",
                F.when(F.col("moneda") == "GTQ", F.lit(1).cast("decimal(10,5)"))
                 .when(F.col("moneda") == "USD", F.col("_tasa_usd")))
    .withColumn("monto_gtq", F.round(F.abs("monto") * F.col("tasa_aplicada"), 2).cast("decimal(18,2)"))
    .withColumn("flag_monto_nulo", F.col("monto").isNull())
    .withColumn("flag_monto_negativo", F.coalesce(F.col("monto") < 0, F.lit(False)))
    .withColumn("flag_usd_sin_tasa", F.coalesce((F.col("moneda") == "USD") & F.col("_tasa_usd").isNull(), F.lit(False)))
    .withColumn("flag_cuenta_inexistente", F.col("_cuenta_existe").isNull())
)

registrar("transacciones", "fecha_hora_no_interpretable", transacciones_gtq.filter(F.col("fecha_hora").isNull()),
          "transaccion_id", "queda NULL, sin tasa asignada")
registrar("transacciones", "moneda_normalizada", transacciones_gtq.filter(F.col("_moneda_orig") != F.col("moneda")),
          "transaccion_id", "upper(trim(moneda))")
registrar("transacciones", "moneda_no_soportada",
          transacciones_gtq.filter(~F.coalesce(F.col("moneda").isin("GTQ", "USD"), F.lit(False))),
          "transaccion_id", "monto_gtq NULL")
registrar("transacciones", "tipo_no_reconocido",
          transacciones_gtq.filter(~F.coalesce(F.col("tipo").isin("CREDITO", "DEBITO"), F.lit(False))),
          "transaccion_id", "se conserva")
registrar("transacciones", "monto_nulo",
          transacciones_gtq.filter(F.col("flag_monto_nulo") & F.expr("nullif(trim(_monto_orig), '') IS NULL")),
          "transaccion_id", "flag_monto_nulo, monto_gtq NULL")
registrar("transacciones", "monto_no_numerico",
          transacciones_gtq.filter(F.col("flag_monto_nulo") & F.expr("nullif(trim(_monto_orig), '') IS NOT NULL")),
          "transaccion_id", "flag_monto_nulo, monto_gtq NULL")
registrar("transacciones", "monto_negativo", transacciones_gtq.filter("flag_monto_negativo"),
          "transaccion_id", "flag_monto_negativo, monto_gtq con valor absoluto")
registrar("transacciones", "cuenta_inexistente", transacciones_gtq.filter("flag_cuenta_inexistente"),
          "transaccion_id", "se conserva con flag_cuenta_inexistente")
registrar("transacciones", "usd_sin_tasa", transacciones_gtq.filter("flag_usd_sin_tasa"),
          "transaccion_id", "flag_usd_sin_tasa, monto_gtq NULL")

transacciones_gtq = transacciones_gtq.select(
    "transaccion_id", "cuenta_id", "fecha_hora", "fecha", "tipo", "canal", "estado",
    "moneda", "monto", "tasa_aplicada", "monto_gtq",
    "flag_monto_nulo", "flag_monto_negativo", "flag_usd_sin_tasa", "flag_cuenta_inexistente",
)
display(transacciones_gtq.orderBy("transaccion_id"))

transaccion_id,cuenta_id,fecha_hora,fecha,tipo,canal,estado,moneda,monto,tasa_aplicada,monto_gtq,flag_monto_nulo,flag_monto_negativo,flag_usd_sin_tasa,flag_cuenta_inexistente
T001,101,2025-06-01T10:15:00.000Z,2025-06-01,CREDITO,APP,COMPLETADA,GTQ,500.00,1.00000,500.00,false,false,false,false
T002,101,2025-06-02T14:30:00.000Z,2025-06-02,DEBITO,TARJETA,COMPLETADA,USD,120.50,7.67568,924.92,false,false,false,false
T003,102,2025-06-03T09:00:00.000Z,2025-06-03,DEBITO,APP,COMPLETADA,GTQ,-80.00,1.00000,80.00,false,true,false,false
T004,103,2025-06-03T18:45:00.000Z,2025-06-03,CREDITO,TRANSFERENCIA,COMPLETADA,GTQ,1500.00,1.00000,1500.00,false,false,false,false
T005,103,2025-06-04T11:20:00.000Z,2025-06-04,DEBITO,APP,RECHAZADA,USD,200.00,7.67883,1535.77,false,false,false,false
T006,105,2025-06-05T16:00:00.000Z,2025-06-05,DEBITO,TARJETA,COMPLETADA,GTQ,null,1.00000,null,true,false,false,false
T007,101,2025-06-10T08:10:00.000Z,2025-06-10,DEBITO,APP,COMPLETADA,USD,75.25,7.68286,578.14,false,false,false,false


## Escritura en silver y reporte de calidad

In [0]:
for nombre, df in (("clientes", clientes), ("cuentas", cuentas), ("transacciones_gtq", transacciones_gtq)):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.{nombre}")

spark.sql(f"COMMENT ON TABLE {SILVER}.transacciones_gtq IS "
          "'Transacciones sin duplicados, con monto en quetzales y flags de calidad. Grano: una transacción.'")

dq_reporte = (
    spark.createDataFrame(hallazgos, "tabla STRING, regla STRING, registros INT, ids STRING, accion STRING")
    .withColumn("fecha_ejecucion", F.current_timestamp())
)
dq_reporte.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.dq_reporte")
display(spark.table(f"{SILVER}.dq_reporte").orderBy(F.desc("registros"), "tabla"))

tabla,regla,registros,ids,accion,fecha_ejecucion
clientes,duplicado_exacto,1,2,eliminado,2026-10-05T19:38:45.749Z
clientes,segmento_faltante,1,4,se asigna 'No definido',2026-10-05T19:38:45.749Z
clientes,departamento_normalizado,1,2,initcap(trim(departamento)),2026-10-05T19:38:45.749Z
clientes,fecha_alta_en_formato_dd/MM/yyyy,1,2,convertida a DATE,2026-10-05T19:38:45.749Z
cuentas,cliente_id_inexistente,1,106,se conserva con flag_cliente_inexistente,2026-10-05T19:38:45.749Z
transacciones,monto_nulo,1,T006,"flag_monto_nulo, monto_gtq NULL",2026-10-05T19:38:45.749Z
transacciones,monto_negativo,1,T003,"flag_monto_negativo, monto_gtq con valor absoluto",2026-10-05T19:38:45.749Z
transacciones,moneda_normalizada,1,T007,upper(trim(moneda)),2026-10-05T19:38:45.749Z
transacciones,duplicado_exacto,1,T004,eliminado,2026-10-05T19:38:45.749Z
clientes,fecha_alta_no_interpretable,0,,queda NULL,2026-10-05T19:38:45.749Z


In [0]:
for tabla, llave in (("clientes", "cliente_id"), ("cuentas", "cuenta_id"), ("transacciones_gtq", "transaccion_id")):
    total, distintos = spark.table(f"{SILVER}.{tabla}").selectExpr("count(*)", f"count(DISTINCT {llave})").first()
    assert total == distintos, f"{tabla}: {total} filas pero {distintos} llaves distintas"
    print(f"{SILVER}.{tabla}: {total} filas, llave única")

workspace.silver.clientes: 5 filas, llave única
workspace.silver.cuentas: 6 filas, llave única
workspace.silver.transacciones_gtq: 7 filas, llave única


## Exportación de `transacciones_gtq` y del reporte de calidad

In [0]:
for nombre in ("transacciones_gtq", "dq_reporte"):
    spark.table(f"{SILVER}.{nombre}").toPandas().to_csv(f"{EXPORTS_PATH}/{nombre}.csv", index=False)
display(dbutils.fs.ls(EXPORTS_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/gold/exports/dq_reporte.csv,dq_reporte.csv,1968,1791229133000
dbfs:/Volumes/workspace/gold/exports/tipo_cambio.csv,tipo_cambio.csv,201,1791228849000
dbfs:/Volumes/workspace/gold/exports/tipo_cambio_raw.xml,tipo_cambio_raw.xml,1432,1791228848000
dbfs:/Volumes/workspace/gold/exports/transacciones_gtq.csv,transacciones_gtq.csv,965,1791229133000
